# NEMO ablation: pre/post d-prime and baseline r

Cells are pooled separately for neuronal and glial fish. Benjamini-Hochberg correction is calculated post hoc from each fish's saved p-values. The d-prime and dFF-difference scatters require difference `q < 0.05`; the r-value scatter requires r-value `q < 0.05`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.stats.multitest import multipletests

DATA_ROOT = Path('/nrs/ahrens/Ziqiang/Motor_clamp/NEMO_ablation')
COHORT_DIRS = {'Neuron': 'neuron_fish', 'Glia': 'glia_fish'}
Q_THRESHOLD = 0.05
D_PRIME_LIMIT = (-1, 1)
R_LIMIT = (-1, 1)
DIFF_LIMIT = (-0.1, 0.1)
POINT_COLOR = {'Neuron': '#1f77b4', 'Glia': '#ff7f0e'}

cwd = Path.cwd().resolve()
REPO_ROOT = next(path for path in (cwd, *cwd.parents) if (path / 'notebooks/NEMO_ablation').is_dir())
FIGURE_DIR = REPO_ROOT / 'notebooks/NEMO_ablation'

sns.set(style='ticks', font_scale=1.0)

In [ ]:
def fdr_significant(p_values, valid_mask):
    tested = valid_mask & np.isfinite(p_values)
    significant = np.zeros(valid_mask.shape, dtype=bool)
    significant[tested] = multipletests(
        p_values[tested], alpha=Q_THRESHOLD, method='fdr_bh'
    )[0]
    return significant


def load_cohort(cohort_dir):
    pooled = {key: [] for key in ('d_prime1', 'd_prime2', 'r1', 'r2', 'diff_1', 'diff_2')}
    n_valid = 0
    n_r_significant = 0
    n_diff_significant = 0

    for fish_dir in sorted(path for path in cohort_dir.iterdir() if path.is_dir()):
        with np.load(fish_dir / 'd_prime_ablation_comparison.npz') as dprime_data, \
             np.load(fish_dir / 'r_ablation_comparison.npz') as r_data, \
             np.load(fish_dir / 'dff_diff_ablation_comparison.npz') as diff_data:
            dprime_mask = np.asarray(dprime_data['valid_mask'], dtype=bool)
            r_mask = np.asarray(r_data['valid_mask'], dtype=bool)
            diff_mask = np.asarray(diff_data['valid_mask'], dtype=bool)
            if not (np.array_equal(dprime_mask, r_mask) and np.array_equal(r_mask, diff_mask)):
                raise ValueError(f'valid_mask mismatch: {fish_dir}')

            dprime_mask &= np.isfinite(dprime_data['d_prime1']) & np.isfinite(dprime_data['d_prime2'])
            r_fdr_mask = fdr_significant(r_data['p1'], r_mask)
            significant_mask = (
                r_mask
                & np.isfinite(r_data['r1'])
                & np.isfinite(r_data['r2'])
                & r_fdr_mask
            )
            diff_fdr_mask = fdr_significant(diff_data['p_1'], diff_mask)
            diff_significant_mask = (
                diff_mask
                & np.isfinite(diff_data['diff_1'])
                & np.isfinite(diff_data['diff_2'])
                & diff_fdr_mask
            )
            dprime_significant_mask = dprime_mask & diff_significant_mask

            pooled['d_prime1'].append(dprime_data['d_prime1'][dprime_significant_mask])
            pooled['d_prime2'].append(dprime_data['d_prime2'][dprime_significant_mask])
            pooled['r1'].append(r_data['r1'][significant_mask])
            pooled['r2'].append(r_data['r2'][significant_mask])
            pooled['diff_1'].append(diff_data['diff_1'][diff_significant_mask])
            pooled['diff_2'].append(diff_data['diff_2'][diff_significant_mask])
            n_valid += int(r_mask.sum())
            n_r_significant += int(significant_mask.sum())
            n_diff_significant += int(diff_significant_mask.sum())

    pooled = {key: np.concatenate(value) for key, value in pooled.items()}
    pooled['n_valid'] = n_valid
    pooled['n_r_significant'] = n_r_significant
    pooled['n_diff_significant'] = n_diff_significant
    return pooled


cohort_data = {
    cohort: load_cohort(DATA_ROOT / dirname)
    for cohort, dirname in COHORT_DIRS.items()
}

pd.DataFrame([
    {
        'cohort': cohort,
        'valid cells': data['n_valid'],
        'd-prime cells plotted (difference q_1 < 0.05)': data['d_prime1'].size,
        'r cells plotted (q1 < 0.05)': data['n_r_significant'],
        'dFF-difference cells plotted (q_1 < 0.05)': data['n_diff_significant'],
    }
    for cohort, data in cohort_data.items()
])

In [ ]:
def scatter_panels(x_key, y_key, limits, xlabel, ylabel, output_name, multiplier=1):
    fig, axes = plt.subplots(1, 2, figsize=(6.5, 3), sharex=True, sharey=True)
    for ax, (cohort, data) in zip(axes, cohort_data.items()):
        ax.scatter(
            multiplier * data[x_key], multiplier * data[y_key],
            s=0.4, alpha=0.08, color=POINT_COLOR[cohort],
            edgecolors='none', rasterized=True,
        )
        ax.plot(limits, limits, '--k', lw=0.8)
        ax.axhline(0, color='0.7', lw=0.6)
        ax.axvline(0, color='0.7', lw=0.6)
        ax.set(xlim=limits, ylim=limits, title=f'{cohort} (n={data[x_key].size:,})')
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlabel(xlabel)
    axes[0].set_ylabel(ylabel)
    sns.despine()
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / output_name, bbox_inches='tight', transparent=True, dpi=300)
    plt.show()


scatter_panels(
    'd_prime1', 'd_prime2', D_PRIME_LIMIT,
    '-d-prime before ablation', '-d-prime after ablation',
    'd_prime_ablation_scatter.pdf',
    multiplier=-1,
)

In [ ]:
scatter_panels(
    'r1', 'r2', R_LIMIT,
    'baseline r before ablation', 'baseline r after ablation',
    'baseline_r_ablation_scatter_q1_005.pdf',
)

scatter_panels(
    'diff_1', 'diff_2', DIFF_LIMIT,
    'mean dFF difference before ablation', 'mean dFF difference after ablation',
    'dff_diff_ablation_scatter_q1_005.pdf',
)